# Como funciona: Assistente de Tecnologia (GenAI + RAG)

Este notebook mostra, passo a passo, a lógica por trás do assistente: desde a primeira
chamada a um modelo de linguagem (LLM) até um sistema de **RAG** que responde com base
em documentos e cita a fonte.

> O produto final é um app em Streamlit. Aqui o foco é **explicar como foi construído**.


## 1. Preparação

Usamos a API do **Google Gemini** (plano gratuito) pela biblioteca `google-genai`.
A chave de API é lida de forma segura, sem ficar salva no notebook.


In [ ]:
!pip install -q google-genai


In [ ]:
from getpass import getpass
from google import genai

api_key = getpass("Cole sua chave da API do Gemini: ")
client = genai.Client(api_key=api_key)


Modelos de LLM mudam e, às vezes, ficam sobrecarregados. Para dar robustez, escolhemos
automaticamente o primeiro modelo disponível de uma lista — um padrão de *fallback*.


In [ ]:
MODELO = None
for nome in ["gemini-flash-lite-latest", "gemini-2.5-flash-lite", "gemini-3.5-flash"]:
    try:
        client.models.generate_content(model=nome, contents="oi")
        MODELO = nome
        break
    except Exception:
        continue
print("Modelo de chat:", MODELO)


## 2. Falando com o modelo

Uma chamada simples: enviamos um texto e recebemos a resposta gerada.


In [ ]:
resposta = client.models.generate_content(
    model=MODELO,
    contents="Explique o que é RAG em uma frase."
)
print(resposta.text)


## 3. Embeddings: transformar texto em números

Para buscar trechos relevantes, convertemos texto em **embeddings** — vetores que
representam o significado. Textos com sentido parecido ficam com vetores próximos,
e isso permite medir o quão parecidos dois textos são (similaridade).


In [ ]:
import numpy as np

EMB_MODELO = None
for nome in ["gemini-embedding-001", "gemini-embedding-2"]:
    try:
        client.models.embed_content(model=nome, contents=["teste"])
        EMB_MODELO = nome
        break
    except Exception:
        continue

def embed(textos):
    r = client.models.embed_content(model=EMB_MODELO, contents=textos)
    return [e.values for e in r.embeddings]

def similaridade(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


## 4. A base de conhecimento

O assistente responde com base em documentos. Aqui usamos exemplos sobre tecnologia;
no app, o usuário pode subir o próprio PDF. Cada documento é transformado em vetor
uma única vez — isso é a **indexação**.


In [ ]:
documentos = [
    "RAG (Retrieval-Augmented Generation) conecta um LLM a documentos externos. Antes de responder, busca os trechos mais relevantes e os envia ao modelo, reduzindo alucinacoes e permitindo citar a fonte.",
    "Embeddings sao vetores de numeros que representam o significado de um texto; textos parecidos ficam com vetores proximos.",
    "O mercado de tecnologia valoriza profissionais de IA. Cargos como Engenheiro de Machine Learning e de GenAI/LLM estao em alta.",
    "Python e a linguagem mais usada em IA e ciencia de dados, pela simplicidade e pelas bibliotecas da area.",
]
vetores = embed(documentos)
print(len(documentos), "documentos indexados.")


## 5. Respondendo com RAG

Para cada pergunta: convertemos em vetor, buscamos os trechos mais parecidos e pedimos
ao modelo uma resposta baseada **somente** nesses trechos. Essa instrução é o *grounding*,
que evita o modelo inventar. A resposta volta junto com a fonte.


In [ ]:
def responder(pergunta, k=2):
    emb_pergunta = embed([pergunta])[0]
    notas = [similaridade(emb_pergunta, v) for v in vetores]
    melhores = np.argsort(notas)[::-1][:k]
    trechos = [documentos[i] for i in melhores]

    prompt = (
        "Responda usando SOMENTE os trechos abaixo. "
        "Se a resposta nao estiver neles, diga 'Nao encontrei isso nos documentos.'\n\n"
        + "\n\n".join(trechos)
        + f"\n\nPergunta: {pergunta}"
    )
    resposta = client.models.generate_content(model=MODELO, contents=prompt).text
    return resposta, trechos


## 6. Testando

Uma pergunta coberta pelos documentos e outra que não está — o assistente deve
**admitir quando não sabe** em vez de inventar.


In [ ]:
r, fontes = responder("O que e RAG e por que reduz alucinacoes?")
print(r)
print("\nFonte usada:", fontes[0][:80], "...")


In [ ]:
r, fontes = responder("Qual e a capital da Franca?")
print(r)


## Conclusão

Em poucas etapas construímos um RAG funcional: ele responde com base nos documentos
fornecidos e admite quando a informação não está disponível, reduzindo alucinações.
O app final (Streamlit) embrulha essa lógica numa interface de chat com upload de PDF.

**Próximos passos:** destacar o trecho exato usado na resposta, suportar vários
documentos ao mesmo tempo e melhorar a divisão em pedaços (chunking).
